<a href="https://colab.research.google.com/github/axg1526-create/Weather_Forecaster/blob/main/Kumar_Weather_Forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!apt-get install nodejs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nodejs is already the newest version (24.21.0-1nodesource1).
The following packages were automatically installed and are no longer required:
  libcares2 node-busboy node-cjs-module-lexer node-undici node-xtend
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.


In [28]:
%%shell
# Install a modern, supported Node.js LTS version (e.g., Node.js 22) in Colab
curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
sudo apt-get install -y nodejs
node -v
npm -v

2026-10-13 06:17:51 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://deb.nodesource.com/node_24.x nodistro InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,631 B in 1s (4,709 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (source

In [29]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import pickle
import os

# Load Dataset
df = pd.read_csv('9cf80863-97ea-4ad8-a4ef-6a5f026debbb.csv')

# Preprocess Data
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['Date'])

df['day_of_year'] = df['Date'].dt.dayofyear
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year

# Cyclical encoding for seasonality continuity
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

df['Rain'] = pd.to_numeric(df['Rain'], errors='coerce').fillna(0)
df['Temp Max'] = pd.to_numeric(df['Temp Max'], errors='coerce')
df['Temp Min'] = pd.to_numeric(df['Temp Min'], errors='coerce')
df = df.dropna(subset=['Temp Max', 'Temp Min'])

# Train strictly on <= 2023, Test on 2024
train_df = df[df['year'] <= 2023]
test_df = df[df['year'] == 2024]

features = ['doy_sin', 'doy_cos', 'month']
targets = ['Temp Max', 'Temp Min', 'Rain']

os.makedirs('model_artifacts', exist_ok=True)
print("--- Training Models (1951-2023) & Evaluating on 2024 ---")

for target in targets:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(train_df[features], train_df[target])

    # Save model artifact
    model_filename = f"model_artifacts/{target.lower().replace(' ', '_')}_model.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)

    # Evaluate on 2024 test set
    preds = model.predict(test_df[features])
    mse = mean_squared_error(test_df[target], preds)
    print(f"MSE for {target} in 2024: {mse:.4f}")

print("\nTraining complete. Models saved successfully in /model_artifacts.")

--- Training Models (1951-2023) & Evaluating on 2024 ---
MSE for Temp Max in 2024: 3.4732
MSE for Temp Min in 2024: 3.9038
MSE for Rain in 2024: 90.8252

Training complete. Models saved successfully in /model_artifacts.


In [30]:
%%writefile predict.py
import sys
import pickle
import numpy as np
import pandas as pd
import json
from datetime import datetime

def predict_weather(date_str):
    try:
        dt = datetime.strptime(date_str.strip(), '%d-%m-%y')
    except ValueError as e:
        print(json.dumps({"error": f"Invalid date format. Use DD-MM-YY. Details: {str(e)}"}))
        sys.exit(1)

    day_of_year = dt.timetuple().tm_yday
    month = dt.month

    doy_sin = np.sin(2 * np.pi * day_of_year / 365.25)
    doy_cos = np.cos(2 * np.pi * day_of_year / 365.25)

    X_input = pd.DataFrame([[doy_sin, doy_cos, month]], columns=['doy_sin', 'doy_cos', 'month'])

    results = {}
    targets = ['Temp Max', 'Temp Min', 'Rain']

    for target in targets:
        model_filename = f"model_artifacts/{target.lower().replace(' ', '_')}_model.pkl"
        with open(model_filename, 'rb') as f:
            model = pickle.load(f)
        pred = model.predict(X_input)[0]
        results[target] = float(pred)

    print(json.dumps(results))

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print(json.dumps({"error": "Date argument missing."}))
        sys.exit(1)
    predict_weather(sys.argv[1])

Overwriting predict.py


In [31]:
%%writefile index.js
const { spawn } = require('child_process');
const path = require('path');

// Get date from command line argument (e.g., node index.js 25-12-26)
const inputDate = process.argv[2];

if (!inputDate) {
    console.error("Please provide a date in DD-MM-YY format. Example: node index.js 25-12-26");
    process.exit(1);
}

const pythonScript = path.join(__dirname, 'predict.py');
const pythonProcess = spawn('python3', [pythonScript, inputDate]);

let outputData = '';
let errorData = '';

pythonProcess.stdout.on('data', (data) => {
    outputData += data.toString();
});

pythonProcess.stderr.on('data', (data) => {
    errorData += data.toString();
});

pythonProcess.on('close', (code) => {
    if (code !== 0 || errorData) {
        console.error('Error during prediction:', errorData || outputData);
        return;
    }

    try {
        const result = JSON.parse(outputData.trim());
        if (result.error) {
            console.log(`Error: ${result.error}`);
        } else {
            console.log(`\n--- Weather Forecast for ${inputDate} ---`);
            console.log(`* Maximum Temperature : ${result['Temp Max'].toFixed(2)} °C`);
            console.log(`* Minimum Temperature : ${result['Temp Min'].toFixed(2)} °C`);
            console.log(`* Rainfall            : ${result['Rain'].toFixed(2)} mm`);
            console.log('-----------------------------------------');
        }
    } catch (err) {
        console.error('Failed to parse model output:', outputData);
    }
});

Overwriting index.js


In [32]:
!node index.js 25-12-27


--- Weather Forecast for 25-12-27 ---
* Maximum Temperature : 27.19 °C
* Minimum Temperature : 15.69 °C
* Rainfall            : 0.04 mm
-----------------------------------------


In [33]:
import os
import json
import subprocess
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# 1. Create the visual HTML wrapper layout
html_header = """
<div style="padding: 15px 15px 0px 15px; background-color: #f8fafc; border-radius: 12px 12px 0 0; font-family: system-ui, sans-serif; border: 1px solid #e2e8f0; border-bottom: none; max-width: 400px;">
  <h3 style="color: #0f172a; margin: 0 0 4px 0;">Weather Forecast Interface</h3>
  <p style="font-size: 13px; color: #64748b; margin: 0 0 10px 0;">Enter a date (DD-MM-YY) to run your model script.</p>
</div>
"""

# 2. Native interactive widgets (never disconnects)
date_input = widgets.Text(value='25-12-26', placeholder='DD-MM-YY', description='Date:', layout=widgets.Layout(width='280px'))
predict_btn = widgets.Button(description='Predict Weather', button_style='primary', layout=widgets.Layout(width='140px', margin='0 0 0 10px'))
ui_row = widgets.HBox([date_input, predict_btn], layout=widgets.Layout(padding='0 15px 15px 15px', background_color='#f8fafc', border='1px solid #e2e8f0', border_top='none', border_bottom_radius='12px', max_width='432px'))
output_area = widgets.Output()

# 3. Action function running your original predict.py logic
def on_click_predict(b):
  with output_area:
    clear_output()
    date_val = date_input.value.strip()

    if not date_val:
      print("❌ Error: Please enter a date first.")
      return

    if not os.path.exists('predict.py'):
      print(f"❌ File Error: 'predict.py' not found in current folder ({os.getcwd()}).")
      print("👉 Please upload your predict.py file to the Colab sidebar first!")
      return

    print("🔄 Running model script...")

    try:
      # Run your model exactly like the Node.js setup did
      process = subprocess.run(
          ['python3', 'predict.py', date_val],
          capture_output=True,
          text=True
      )

      # Check for script errors
      if process.returncode != 0:
        print(f"❌ predict.py crashed (Exit Code {process.returncode})")
        print(f"Error Details:\n{process.stderr or process.stdout}")
        return

      # Parse the JSON string from predict.py
      stdout_clean = process.stdout.strip()
      result = json.loads(stdout_clean)

      if "error" in result:
        print(f"❌ Script Error: {result['error']}")
      else:
        # Display the formatted weather card matching your Node setup
        display(HTML(f"""
        <div style="margin-top: 15px; padding: 14px; background-color: #ffffff; border-radius: 8px; border: 1px solid #e2e8f0; max-width: 400px; font-family: monospace; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
          <h4 style="margin: 0 0 10px 0; color: #1e293b; border-bottom: 1px solid #f1f5f9; padding-bottom: 6px; font-family: system-ui, sans-serif;">--- Weather Forecast for {date_val} ---</h4>
          <div style="font-size: 13px; color: #334155; line-height: 1.7;">
            <div>☀️ Maximum Temperature : <span style="font-weight: bold; color: #ea580c;">{float(result['Temp Max']):.2f} °C</span></div>
            <div>🌙 Minimum Temperature : <span style="font-weight: bold; color: #2563eb;">{float(result['Temp Min']):.2f} °C</span></div>
            <div>🌧️ Rainfall            : <span style="font-weight: bold; color: #0891b2;">{float(result['Rain']):.2f} mm</span></div>
          </div>
          <div style="color: #cbd5e1; font-size: 10px; margin-top: 8px;">-----------------------------------------</div>
        </div>
        """))

    except json.JSONDecodeError:
      print("❌ Parsing Error: Failed to decode JSON output from predict.py.")
      print(f"Raw Output Received:\n{process.stdout}")
    except Exception as e:
      print(f"❌ System Error: {str(e)}")

# Link click trigger to the execution function
predict_btn.on_click(on_click_predict)

# Render everything on screen
display(HTML(html_header))
display(ui_row)
display(output_area)


Output()